In [2]:
library(dplyr)
library(readr)

suppressPackageStartupMessages({
  library(rtracklayer)
  library(GenomicRanges)
})

library(data.table)



## Liftover .bed files from hg38 to hg19

In [13]:
library(rtracklayer)
chain <- import.chain("../ref/hg38ToHg19.over.chain")


In [18]:
# Before unlisting, check mapping success
lifted_list <- liftOver(dhs_gr, chain)
success <- lengths(lifted_list) > 0

cat("Successfully mapped:", sum(success), "\n")
cat("Failed to map:", sum(!success), "\n")
cat("Mapping rate:", round(100 * sum(success) / length(dhs_gr), 2), "%\n")

# Get unmapped regions with their metadata
unmapped <- dhs_gr[!success]
if (length(unmapped) > 0) {
    cat("\nFirst few unmapped regions:\n")
    print(head(unmapped))
}

Successfully mapped: 101442 
Failed to map: 128 
Mapping rate: 99.87 %

First few unmapped regions:
GRanges object with 6 ranges and 9 metadata columns:
      seqnames              ranges strand |        name mean_signal peak.count
         <Rle>           <IRanges>  <Rle> | <character>   <numeric>  <integer>
  [1]     chr1 103348549-103348674      * |     1.47353   0.3613340          1
  [2]     chr1 122549860-122549946      * |    1.542971   0.0651759          3
  [3]     chr1 122924140-122924279      * |    1.544275   0.1735307          3
  [4]     chr1 123265809-123265918      * |     1.54551   0.0956473          1
  [5]     chr1 123591143-123591213      * |    1.546677   0.1029033          7
  [6]     chr1 124656668-124656750      * |    1.550564   0.0402407          2
         summit core_start  core_end   component  position      score
      <integer>  <numeric> <numeric> <character> <integer>  <numeric>
  [1] 103348610  103348610 103348610     Cardiac 103348610 0.01135557
  [2]

## Updated full metadata liftover 

## DHS

In [9]:
#!/usr/bin/env Rscript
# ================================================================
# DHS Liftover Script (hg38 → hg19) — Relative summit/core offsets
# ------------------------------------------------
#  - Computes summit/core offsets relative to peak start
#  - Lifts only the DHS peak ranges
#  - Reconstructs summit/core positions in hg19 using offsets
#  - Keeps only 1:1 mappings
# ================================================================

suppressPackageStartupMessages({
  library(rtracklayer)
  library(GenomicRanges)
  library(data.table)
})

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------
input_dir  <- "/dcs07/scharpf/data/jzavras/LUCAS_Olink/Lucas-Cancer-Screening/scripts/lung-cancer-screening-paper/methods_code/WG/ref/PEARL/data/Bed_files_hg38/DHS_sites/"
output_dir <- "../ref/CRE_sites/DHS_sites2"
chain_path <- "../ref/hg38ToHg19.over.chain"

dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
chain <- import.chain(chain_path)

# ------------------------------------------------
# 2. Process each file
# ------------------------------------------------
bed_files <- list.files(input_dir, pattern = "\\.bed$", full.names = TRUE)
cat("Found", length(bed_files), "BED files to process\n\n")

summary_stats <- data.frame(
  File = character(),
  Total_Regions = integer(),
  Successfully_Mapped = integer(),
  Failed_to_Map = integer(),
  Multi_Mapped_Dropped = integer(),
  Mapping_Rate_Percent = numeric(),
  stringsAsFactors = FALSE
)

for (bed_file in bed_files) {
  file_name <- basename(bed_file)
  cat("\nProcessing:", file_name, "\n")

  dhs_df <- fread(bed_file, sep="\t", header=TRUE, fill=TRUE)
  total_regions <- nrow(dhs_df)

  # --- compute relative offsets (hg38) ---
  dhs_df$summit_offset      <- dhs_df$summit      - dhs_df$Start
  dhs_df$core_start_offset  <- dhs_df$core_start  - dhs_df$Start
  dhs_df$core_end_offset    <- dhs_df$core_end    - dhs_df$Start

  # --- create GRanges for peak ranges ---
  dhs_gr <- GRanges(
    seqnames = dhs_df$Chrom,
    ranges   = IRanges(start = dhs_df$Start, end = dhs_df$End),
    strand   = "*"
  )

  # --- liftover ranges ---
  lifted_list <- liftOver(dhs_gr, chain)

  # keep 1:1 mappings only
  single_hits <- which(lengths(lifted_list) == 1)
  multi_hits  <- which(lengths(lifted_list) > 1)
  if (length(multi_hits) > 0)
    cat("  Note:", length(multi_hits), "multi-mapped DHS sites dropped\n")

  lifted <- unlist(lifted_list[single_hits])
  dhs_df <- dhs_df[single_hits, ]

  mapped <- length(single_hits)
  failed <- total_regions - mapped
  mapping_rate <- round(100 * mapped / total_regions, 2)

  cat("  Successfully mapped:", mapped, "\n")
  cat("  Failed to map:", failed, "\n")
  cat("  Mapping rate:", mapping_rate, "%\n")

  # --- reconstruct summit/core positions in hg19 ---
  lifted_df <- as.data.frame(lifted)
  lifted_df <- data.table(
    Chrom = lifted_df$seqnames,
    Start = lifted_df$start,
    End   = lifted_df$end
  )

  # Use relative offsets to compute hg19 positions
  lifted_df[, summit_hg19     := Start + dhs_df$summit_offset]
  lifted_df[, core_start_hg19 := Start + dhs_df$core_start_offset]
  lifted_df[, core_end_hg19   := Start + dhs_df$core_end_offset]

  # Merge metadata
  lifted_df[, `:=`(
    name = dhs_df$name,
    mean_signal = dhs_df$mean_signal,
    peak.count = dhs_df$peak.count,
    component = dhs_df$component,
    position = dhs_df$position,
    score = dhs_df$score
  )]

  # --- save output ---
  output_file <- file.path(output_dir, file_name)
  fwrite(lifted_df, output_file, sep="\t", quote=FALSE)
  cat("  ✅ Saved:", output_file, "\n")

  # --- record summary ---
  summary_stats <- rbind(summary_stats, data.frame(
    File = file_name,
    Total_Regions = total_regions,
    Successfully_Mapped = mapped,
    Failed_to_Map = failed,
    Multi_Mapped_Dropped = length(multi_hits),
    Mapping_Rate_Percent = mapping_rate
  ))
}

# ------------------------------------------------
# 3. Save summary
# ------------------------------------------------
summary_file <- file.path(output_dir, "liftover_summary.txt")
fwrite(summary_stats, summary_file, sep="\t", quote=FALSE)

cat("\n✅ All files processed!\n")
cat("Summary saved to:", summary_file, "\n")

cat("\n=== Overall Summary ===\n")
cat("Total files processed:", nrow(summary_stats), "\n")
cat("Total regions:", sum(summary_stats$Total_Regions), "\n")
cat("Successfully mapped:", sum(summary_stats$Successfully_Mapped), "\n")
cat("Failed to map:", sum(summary_stats$Failed_to_Map), "\n")
cat("Multi-mapped dropped:", sum(summary_stats$Multi_Mapped_Dropped), "\n")
cat("Overall mapping rate:",
    round(100 * sum(summary_stats$Successfully_Mapped) / sum(summary_stats$Total_Regions), 2),
    "%\n")


Found 16 BED files to process


Processing: Cancer_epithelial.bed 
  Note: 160 multi-mapped DHS sites dropped
  Successfully mapped: 126641 
  Failed to map: 568 
  Mapping rate: 99.55 %
  ✅ Saved: ../ref/CRE_sites/DHS_sites2/Cancer_epithelial.bed 

Processing: Cardiac.bed 
  Note: 102 multi-mapped DHS sites dropped
  Successfully mapped: 101340 
  Failed to map: 230 
  Mapping rate: 99.77 %
  ✅ Saved: ../ref/CRE_sites/DHS_sites2/Cardiac.bed 

Processing: Digestive.bed 
  Note: 143 multi-mapped DHS sites dropped
  Successfully mapped: 114144 
  Failed to map: 432 
  Mapping rate: 99.62 %
  ✅ Saved: ../ref/CRE_sites/DHS_sites2/Digestive.bed 

Processing: Lymphoid.bed 
  Note: 514 multi-mapped DHS sites dropped
  Successfully mapped: 190911 
  Failed to map: 1461 
  Mapping rate: 99.24 %
  ✅ Saved: ../ref/CRE_sites/DHS_sites2/Lymphoid.bed 

Processing: Musculoskeletal.bed 
  Note: 172 multi-mapped DHS sites dropped
  Successfully mapped: 181336 
  Failed to map: 786 
  Mapping rate: 99.5

In [11]:
library(data.table)

# Replace with the actual output file you just generated
dhs <- fread("../ref/CRE_sites/DHS_sites2/Cardiac.bed")

# Check how many summit_hg19 positions fall within their peak ranges
inside_fraction <- sum(dhs$summit_hg19 >= dhs$Start & dhs$summit_hg19 <= dhs$End, na.rm = TRUE) / nrow(dhs)

cat(sprintf("\nFraction of summits within their DHS range: %.3f\n", inside_fraction))



Fraction of summits within their DHS range: 1.000


## ATAC

In [12]:
#!/usr/bin/env Rscript
# ================================================================
# ATAC Liftover Script (hg38 → hg19) — Relative summit offsets
# ------------------------------------------------
#  - Computes summit offset (bp from peak Start)
#  - Lifts only ATAC ranges (1:1 mappings only)
#  - Reconstructs ATAC_summit_hg19 using the offset
#  - Preserves all metadata; writes summary
# ================================================================

suppressPackageStartupMessages({
  library(rtracklayer)
  library(GenomicRanges)
  library(data.table)
})

# ------------------------------------------------
# 1) Paths
# ------------------------------------------------
input_dir  <- "/dcs07/scharpf/data/jzavras/LUCAS_Olink/Lucas-Cancer-Screening/scripts/lung-cancer-screening-paper/methods_code/WG/ref/PEARL/data/Bed_files_hg38/scATAC_sites/"
output_dir <- "../ref/CRE_sites/scATAC_sites2/"
chain_path <- "../ref/hg38ToHg19.over.chain"

dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
chain <- import.chain(chain_path)

# ------------------------------------------------
# 2) Utilities
# ------------------------------------------------
# choose summit as absolute hg38 coordinate, then derive offset vs Start
compute_summit_abs <- function(df) {
  if ("position" %in% names(df) && is.numeric(df$position)) {
    s_abs <- as.numeric(df$position) # absolute hg38
  } else if ("summit_distance" %in% names(df) && is.numeric(df$summit_distance)) {
    s_abs <- as.numeric(df$Start) + as.numeric(df$summit_distance) # absolute from Start + distance
  } else {
    # fallback: midpoint of hg38 peak
    s_abs <- as.numeric(df$Start) + floor((as.numeric(df$End) - as.numeric(df$Start)) / 2)
  }
  # ensure numeric
  s_abs[!is.finite(s_abs)] <- NA_real_
  s_abs
}

# ------------------------------------------------
# 3) Process all ATAC files
# ------------------------------------------------
bed_files <- list.files(input_dir, pattern="\\.bed$", full.names=TRUE)
cat("Found", length(bed_files), "ATAC BED files\n\n")

summary_stats <- data.frame(
  File = character(),
  Total_Regions = integer(),
  Single_Mapped = integer(),
  Failed_or_Dropped = integer(),
  Multi_Mapped_Dropped = integer(),
  Summit_Inside_Range_hg38 = numeric(),
  Summit_Inside_Range_hg19 = numeric(),
  stringsAsFactors = FALSE
)

for (bed_file in bed_files) {
  file_name <- basename(bed_file)
  cat("\nProcessing:", file_name, "\n")

  # ---- Load ----
  atac_df <- fread(bed_file, sep="\t", header=TRUE, fill=TRUE)
  total_regions <- nrow(atac_df)

  # Basic column checks
  needed <- c("Chrom","Start","End")
  if (!all(needed %in% names(atac_df))) {
    stop("Bed file missing required columns Chrom/Start/End: ", file_name)
  }

  # ---- Compute absolute hg38 summit + offset ----
  atac_df$summit_hg38 <- compute_summit_abs(atac_df)
  atac_df$summit_offset <- atac_df$summit_hg38 - atac_df$Start

  # Summit-in-range (hg38) QC (fraction)
  frac_hg38 <- sum(atac_df$summit_hg38 >= atac_df$Start &
                   atac_df$summit_hg38 <= atac_df$End, na.rm=TRUE) / total_regions

  # ---- Build GRanges for ranges (hg38) ----
  atac_gr <- GRanges(
    seqnames = atac_df$Chrom,
    ranges   = IRanges(start = atac_df$Start, end = atac_df$End),
    strand   = "*"
  )

  # ---- LiftOver ranges → keep only 1:1 mappings ----
  lifted_list <- liftOver(atac_gr, chain)
  single_hits <- which(lengths(lifted_list) == 1)
  multi_hits  <- which(lengths(lifted_list) > 1)
  if (length(multi_hits) > 0)
    cat("  Note:", length(multi_hits), "multi-mapped ATAC peaks dropped\n")

  lifted <- unlist(lifted_list[single_hits])
  # Align metadata to 1:1 subset
  atac_df <- atac_df[single_hits, ]

  single_mapped <- length(single_hits)
  failed_or_dropped <- total_regions - single_mapped

  cat("  Single-mapped (1:1):", single_mapped, "\n")
  cat("  Failed or dropped  :", failed_or_dropped, "\n")

  # ---- Reconstruct ATAC_summit_hg19 using offset ----
  lifted_dt <- as.data.table(as.data.frame(lifted))[, .(Chrom=seqnames, Start=start, End=end)]
  # use the previously computed offset
  lifted_dt[, ATAC_summit_hg19 := Start + atac_df$summit_offset]

  # Sanity: summit within hg19 range fraction
  frac_hg19 <- sum(lifted_dt$ATAC_summit_hg19 >= lifted_dt$Start &
                   lifted_dt$ATAC_summit_hg19 <= lifted_dt$End, na.rm=TRUE) / nrow(lifted_dt)

  # ---- Assemble final output (coords first, then metadata) ----
  keep_meta <- setdiff(names(atac_df), c("Chrom","Start","End"))  # preserve all other columns
  out_dt <- cbind(
    lifted_dt,
    atac_df[, ..keep_meta]
  )

  # ---- Write file ----
  out_path <- file.path(output_dir, file_name)
  fwrite(out_dt, out_path, sep="\t", quote=FALSE)
  cat("  ✅ Saved:", out_path, "\n")
  cat(sprintf("  Summit-in-range (hg38): %.3f | (hg19): %.3f\n", frac_hg38, frac_hg19))

  # ---- Summarize ----
  summary_stats <- rbind(summary_stats, data.frame(
    File = file_name,
    Total_Regions = total_regions,
    Single_Mapped = single_mapped,
    Failed_or_Dropped = failed_or_dropped,
    Multi_Mapped_Dropped = length(multi_hits),
    Summit_Inside_Range_hg38 = round(frac_hg38, 3),
    Summit_Inside_Range_hg19 = round(frac_hg19, 3)
  ))
}

# ------------------------------------------------
# 4) Save summary
# ------------------------------------------------
summary_file <- file.path(output_dir, "liftover_summary.txt")
fwrite(summary_stats, summary_file, sep="\t", quote=FALSE)

cat("\n✅ All ATAC files processed!\n")
cat("Summary written to:", summary_file, "\n")


Found 20 ATAC BED files


Processing: Adrenal_Cortical.bed 


Discarding unchained sequences: chr7_KI270803v1_alt



  Note: 77 multi-mapped ATAC peaks dropped
  Single-mapped (1:1): 36279 
  Failed or dropped  : 110 
  ✅ Saved: ../ref/CRE_sites/scATAC_sites2//Adrenal_Cortical.bed 
  Summit-in-range (hg38): 1.000 | (hg19): 1.000

Processing: Adult_Stromal.bed 
  Note: 310 multi-mapped ATAC peaks dropped
  Single-mapped (1:1): 70664 
  Failed or dropped  : 1199 
  ✅ Saved: ../ref/CRE_sites/scATAC_sites2//Adult_Stromal.bed 
  Summit-in-range (hg38): 1.000 | (hg19): 0.999

Processing: Cardiomyocyte.bed 


Discarding unchained sequences: chr22_KI270879v1_alt



  Note: 33 multi-mapped ATAC peaks dropped
  Single-mapped (1:1): 22518 
  Failed or dropped  : 46 
  ✅ Saved: ../ref/CRE_sites/scATAC_sites2//Cardiomyocyte.bed 
  Summit-in-range (hg38): 1.000 | (hg19): 1.000

Processing: Endothelial.bed 


...

  Note: 65 multi-mapped ATAC peaks dropped
  Single-mapped (1:1): 36383 
  Failed or dropped  : 99 
  ✅ Saved: ../ref/CRE_sites/scATAC_sites2//Pulm_Epithelial.bed 
  Summit-in-range (hg38): 1.000 | (hg19): 1.000

Processing: Skeletal_Myocyte.bed 


Discarding unchained sequences: chr7_KI270803v1_alt



  Note: 64 multi-mapped ATAC peaks dropped
  Single-mapped (1:1): 27840 
  Failed or dropped  : 79 
  ✅ Saved: ../ref/CRE_sites/scATAC_sites2//Skeletal_Myocyte.bed 
  Summit-in-range (hg38): 1.000 | (hg19): 1.000

Processing: Skin_Mammary_Epithelial.bed 
  Note: 111 multi-mapped ATAC peaks dropped
  Single-mapped (1:1): 48156 
  Failed or dropped  : 180 
  ✅ Saved: ../ref/CRE_sites/scATAC_sites2//Skin_Mammary_Epithelial.bed 
  Summit-in-range (hg38): 1.000 | (hg19): 1.000

Processing: Stromal.bed 


Discarding unchained sequences: chr22_KI270879v1_alt, chr7_KI270803v1_alt, chr8_KI270821v1_alt, chr15_KI270850v1_alt, chr4_GL000008v2_random, chrUn_KI270742v1



  Note: 145 multi-mapped ATAC peaks dropped
  Single-mapped (1:1): 61860 
  Failed or dropped  : 243 
  ✅ Saved: ../ref/CRE_sites/scATAC_sites2//Stromal.bed 
  Summit-in-range (hg38): 1.000 | (hg19): 1.000

Processing: Ubiquitous.bed 
  Note: 282 multi-mapped ATAC peaks dropped
  Single-mapped (1:1): 125989 
  Failed or dropped  : 282 
  ✅ Saved: ../ref/CRE_sites/scATAC_sites2//Ubiquitous.bed 
  Summit-in-range (hg38): 1.000 | (hg19): 1.000

✅ All ATAC files processed!
Summary written to: ../ref/CRE_sites/scATAC_sites2//liftover_summary.txt 


## TCGA

In [4]:
#!/usr/bin/env Rscript
# ================================================================
# TCGA ATAC-seq Liftover Script (hg38 → hg19)
# ------------------------------------------------
#  - Keeps only 1:1 mappings
#  - Preserves metadata
#  - Cleans output filenames
#  - Outputs BEDs to ../ref/CRE_sites/TCGA_sites2/
# ================================================================

suppressPackageStartupMessages({
  library(rtracklayer)
  library(GenomicRanges)
  library(data.table)
})

# ------------------------------------------------
# 1. Paths
# ------------------------------------------------
input_dir  <- "../ref/TCGA/"
output_dir <- "../ref/CRE_sites/TCGA_sites2/"
chain_path <- "../ref/hg38ToHg19.over.chain"

dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
chain <- import.chain(chain_path)

# ------------------------------------------------
# 2. Process each TCGA .txt file
# ------------------------------------------------
txt_files <- list.files(input_dir, pattern = "\\.txt$", full.names = TRUE)
cat("Found", length(txt_files), "files to process\n\n")

summary_stats <- data.frame(
  File = character(),
  Total_Regions = integer(),
  Successfully_Mapped = integer(),
  Failed_to_Map = integer(),
  Multi_Mapped_Dropped = integer(),
  Mapping_Rate_Percent = numeric(),
  stringsAsFactors = FALSE
)

for (txt_file in txt_files) {
  file_name <- basename(txt_file)
  # Clean cancer type from filename
  cancer_type <- gsub("_peakCalls.*|_ATAC.*|\\.txt$", "", file_name)
  cat("\nProcessing:", file_name, "\n")

  # --- load ---
  df <- fread(txt_file, sep = "\t", header = TRUE)
  total_regions <- nrow(df)

  # --- create GRanges ---
  gr <- GRanges(
    seqnames = df$seqnames,
    ranges   = IRanges(start = df$start, end = df$end),
    strand   = "*"
  )
  meta_cols <- setdiff(names(df), c("seqnames", "start", "end"))
  mcols(gr) <- as.data.frame(df[, ..meta_cols])

  # --- liftover ---
  lifted_list <- liftOver(gr, chain)

  # keep only 1:1 mappings
  single_hits <- which(lengths(lifted_list) == 1)
  multi_hits  <- which(lengths(lifted_list) > 1)
  if (length(multi_hits) > 0)
    cat("  Note:", length(multi_hits), "multi-mapped sites dropped\n")

  lifted <- unlist(lifted_list[single_hits])
  df <- df[single_hits, ]

  mapped <- length(single_hits)
  failed <- total_regions - mapped
  mapping_rate <- round(100 * mapped / total_regions, 2)

  cat("  Successfully mapped:", mapped, "\n")
  cat("  Failed:", failed, "\n")
  cat("  Mapping rate:", mapping_rate, "%\n")

  # --- convert to BED-like table ---
  lifted_df <- data.frame(
    Chrom = as.character(seqnames(lifted)),
    Start = start(lifted),
    End   = end(lifted),
    name  = if ("name" %in% colnames(mcols(lifted))) mcols(lifted)$name else paste0(cancer_type, "_", seq_along(lifted)),
    score = if ("score" %in% colnames(mcols(lifted))) mcols(lifted)$score else ".",
    annotation = if ("annotation" %in% colnames(mcols(lifted))) mcols(lifted)$annotation else ".",
    percentGC = if ("percentGC" %in% colnames(mcols(lifted))) mcols(lifted)$percentGC else NA,
    percentAT = if ("percentAT" %in% colnames(mcols(lifted))) mcols(lifted)$percentAT else NA
  )

  # --- clean filename for output ---
  out_name <- paste0(cancer_type, "_ATAC_Peaks_hg19.bed")
  output_file <- file.path(output_dir, out_name)

  # --- save BED output ---
  fwrite(lifted_df, output_file, sep = "\t", quote = FALSE)
  cat("  ✅ Saved:", output_file, "\n")

  # --- record summary ---
  summary_stats <- rbind(summary_stats, data.frame(
    File = file_name,
    Total_Regions = total_regions,
    Successfully_Mapped = mapped,
    Failed_to_Map = failed,
    Multi_Mapped_Dropped = length(multi_hits),
    Mapping_Rate_Percent = mapping_rate
  ))
}

# ------------------------------------------------
# 3. Save summary
# ------------------------------------------------
summary_file <- file.path(output_dir, "liftover_summary.txt")
fwrite(summary_stats, summary_file, sep = "\t", quote = FALSE)

cat("\n✅ All files processed!\n")
cat("Summary saved to:", summary_file, "\n")

cat("\n=== Overall Summary ===\n")
cat("Total files processed:", nrow(summary_stats), "\n")
cat("Total regions:", sum(summary_stats$Total_Regions), "\n")
cat("Successfully mapped:", sum(summary_stats$Successfully_Mapped), "\n")
cat("Failed to map:", sum(summary_stats$Failed_to_Map), "\n")
cat("Multi-mapped dropped:", sum(summary_stats$Multi_Mapped_Dropped), "\n")
cat("Overall mapping rate:",
    round(100 * sum(summary_stats$Successfully_Mapped) / sum(summary_stats$Total_Regions), 2),
    "%\n")


Found 23 files to process


Processing: ACC_peakCalls.txt 
  Note: 290 multi-mapped sites dropped
  Successfully mapped: 90373 
  Failed: 404 
  Mapping rate: 99.55 %
  ✅ Saved: ../ref/CRE_sites/TCGA_sites2//ACC_ATAC_Peaks_hg19.bed 

Processing: BLCA_peakCalls.txt 
  Note: 354 multi-mapped sites dropped
  Successfully mapped: 107944 
  Failed: 529 
  Mapping rate: 99.51 %
  ✅ Saved: ../ref/CRE_sites/TCGA_sites2//BLCA_ATAC_Peaks_hg19.bed 

Processing: BRCA_peakCalls.txt 
  Note: 627 multi-mapped sites dropped
  Successfully mapped: 214841 
  Failed: 1137 
  Mapping rate: 99.47 %
  ✅ Saved: ../ref/CRE_sites/TCGA_sites2//BRCA_ATAC_Peaks_hg19.bed 

Processing: CESC_peakCalls.txt 
  Note: 197 multi-mapped sites dropped
  Successfully mapped: 55825 
  Failed: 300 
  Mapping rate: 99.47 %
  ✅ Saved: ../ref/CRE_sites/TCGA_sites2//CESC_ATAC_Peaks_hg19.bed 

Processing: CHOL_peakCalls.txt 
  Note: 253 multi-mapped sites dropped
  Successfully mapped: 67724 
  Failed: 402 
  Mapping rate: 99.41 %

## QC

In [13]:
##### QC of the Liftover #####

library(rtracklayer)
library(GenomicRanges)
library(data.table)

# === 1. Load hg19 lifted files ===
atac_hg19_file <- "../ref/CRE_sites/scATAC_sites2/Adrenal_Cortical.bed"
dhs_hg19_file  <- "../ref/CRE_sites/DHS_sites2/Cardiac.bed"

atac_hg19_df <- fread(atac_hg19_file)
dhs_hg19_df  <- fread(dhs_hg19_file)

atac_hg19_gr <- GRanges(seqnames = atac_hg19_df$Chrom,
                        ranges = IRanges(start = atac_hg19_df$Start, end = atac_hg19_df$End))
dhs_hg19_gr  <- GRanges(seqnames = dhs_hg19_df$Chrom,
                        ranges = IRanges(start = dhs_hg19_df$Start, end = dhs_hg19_df$End))

# === 2. Load original hg38 files ===
atac_hg38_file <- "/dcs07/scharpf/data/jzavras/LUCAS_Olink/Lucas-Cancer-Screening/scripts/lung-cancer-screening-paper/methods_code/WG/ref/PEARL/data/Bed_files_hg38/scATAC_sites/Adrenal_Cortical.bed"
dhs_hg38_file  <- "/dcs07/scharpf/data/jzavras/LUCAS_Olink/Lucas-Cancer-Screening/scripts/lung-cancer-screening-paper/methods_code/WG/ref/PEARL/data/Bed_files_hg38/DHS_sites/Cardiac.bed"

atac_hg38_df <- fread(atac_hg38_file)
dhs_hg38_df  <- fread(dhs_hg38_file)

atac_hg38_gr <- GRanges(seqnames = atac_hg38_df$Chrom,
                        ranges = IRanges(start = atac_hg38_df$Start, end = atac_hg38_df$End))
dhs_hg38_gr  <- GRanges(seqnames = dhs_hg38_df$Chrom,
                        ranges = IRanges(start = dhs_hg38_df$Start, end = dhs_hg38_df$End))

cat("=== BASIC STATISTICS ===\n\n")

# --- ATAC ---
cat("ATAC-seq:\n")
cat("  hg38 regions:", length(atac_hg38_gr), "\n")
cat("  hg19 regions:", length(atac_hg19_gr), "\n")
cat("  Retention rate:", round(100 * length(atac_hg19_gr) / length(atac_hg38_gr), 2), "%\n\n")

# --- DHS ---
cat("DHS:\n")
cat("  hg38 regions:", length(dhs_hg38_gr), "\n")
cat("  hg19 regions:", length(dhs_hg19_gr), "\n")
cat("  Retention rate:", round(100 * length(dhs_hg19_gr) / length(dhs_hg38_gr), 2), "%\n\n")

cat("=== WIDTH DISTRIBUTION ===\n\n")
cat("ATAC widths (mean/median):",
    round(mean(width(atac_hg38_gr)),1),"/",median(width(atac_hg38_gr)),"→",
    round(mean(width(atac_hg19_gr)),1),"/",median(width(atac_hg19_gr)),"\n")
cat("DHS  widths (mean/median):",
    round(mean(width(dhs_hg38_gr)),1),"/",median(width(dhs_hg38_gr)),"→",
    round(mean(width(dhs_hg19_gr)),1),"/",median(width(dhs_hg19_gr)),"\n\n")

cat("=== SUMMIT-IN-RANGE QC ===\n\n")
# check that lifted summits lie within lifted ranges
if ("ATAC_summit_hg19" %in% names(atac_hg19_df))
  cat("ATAC summits within range:",
      sum(atac_hg19_df$ATAC_summit_hg19 >= atac_hg19_df$Start &
          atac_hg19_df$ATAC_summit_hg19 <= atac_hg19_df$End, na.rm=TRUE) /
        nrow(atac_hg19_df), "\n")
if ("summit_hg19" %in% names(dhs_hg19_df))
  cat("DHS summits within range:",
      sum(dhs_hg19_df$summit_hg19 >= dhs_hg19_df$Start &
          dhs_hg19_df$summit_hg19 <= dhs_hg19_df$End, na.rm=TRUE) /
        nrow(dhs_hg19_df), "\n")

cat("\n=== CHROMOSOME DISTRIBUTION ===\n\n")
cat("ATAC hg19:\n"); print(table(seqnames(atac_hg19_gr)))
cat("DHS hg19:\n");  print(table(seqnames(dhs_hg19_gr)))

cat("\n=== METADATA PRESERVATION ===\n\n")
cat("ATAC columns identical:",
    all(names(atac_hg38_df) %in% names(atac_hg19_df)), "\n")
cat("DHS  columns identical:",
    all(names(dhs_hg38_df) %in% names(dhs_hg19_df)), "\n")

cat("\n=== SAMPLE REGION CHECK ===\n\n")
cat("ATAC first region hg38:",
    as.character(seqnames(atac_hg38_gr[1])),":",start(atac_hg38_gr[1]),"-",end(atac_hg38_gr[1]),"\n")
cat("ATAC first region hg19:",
    as.character(seqnames(atac_hg19_gr[1])),":",start(atac_hg19_gr[1]),"-",end(atac_hg19_gr[1]),"\n")
cat("DHS  first region hg38:",
    as.character(seqnames(dhs_hg38_gr[1])),":",start(dhs_hg38_gr[1]),"-",end(dhs_hg38_gr[1]),"\n")
cat("DHS  first region hg19:",
    as.character(seqnames(dhs_hg19_gr[1])),":",start(dhs_hg19_gr[1]),"-",end(dhs_hg19_gr[1]),"\n")

cat("\n=== QC SUMMARY ===\n")


=== BASIC STATISTICS ===

ATAC-seq:
  hg38 regions: 36389 
  hg19 regions: 36279 
  Retention rate: 99.7 %

DHS:
  hg38 regions: 101570 
  hg19 regions: 101340 
  Retention rate: 99.77 %

=== WIDTH DISTRIBUTION ===

ATAC widths (mean/median): 401 / 401 → 401 / 401 
DHS  widths (mean/median): 206.6 / 199 → 206.6 / 199 

=== SUMMIT-IN-RANGE QC ===

ATAC summits within range: 0.9999173 
DHS summits within range: 0.9999605 

=== CHROMOSOME DISTRIBUTION ===

ATAC hg19:

 chr4  chrX  chr2  chr1 chr11 chr10  chr3 chr16 chr12  chr7  chr5 chr19 chr13 
 1659  1047  3083  3228  1795  1644  2526  1274  1850  1922  1883   932  1109 
chr20  chr9  chr6 chr22 chr17 chr15  chr8 chr18 chr14 chr21 
 1106  1645  1969   704  1379  1167  1788   896  1242   431 
DHS hg19:

 chr1 chr10 chr11 chr12 chr13 chr14 chr15 chr16 chr17 chr18 chr19  chr2 chr20 
 9267  5314  4844  4729  3383  2878  3513  2415  2842  2914  1472  8843  2075 
chr21 chr22  chr3  chr4  chr5  chr6  chr7  chr8  chr9  chrX  chrY 
 1036  1373  7

In [5]:
#!/usr/bin/env Rscript
# ================================================================
# TCGA Liftover QC — validate hg38 → hg19 conversion
# ================================================================

suppressPackageStartupMessages({
  library(rtracklayer)
  library(GenomicRanges)
  library(data.table)
})

# ------------------------------------------------
# 1. Paths and file setup
# ------------------------------------------------
input_hg38_dir <- "../ref/TCGA/"
input_hg19_dir <- "../ref/CRE_sites/TCGA_sites2/"

# Automatically find one-to-one pairs based on cancer type
hg38_files <- list.files(input_hg38_dir, pattern = "_peakCalls\\.txt$", full.names = TRUE)
hg19_files <- list.files(input_hg19_dir, pattern = "_ATAC_Peaks_hg19\\.bed$", full.names = TRUE)

cancer_types <- gsub("_peakCalls\\.txt$", "", basename(hg38_files))
cat("Found", length(cancer_types), "cancer types for QC\n\n")

# ------------------------------------------------
# 2. Initialize summary
# ------------------------------------------------
qc_summary <- data.frame(
  Cancer_Type = character(),
  hg38_Regions = integer(),
  hg19_Regions = integer(),
  Retention_Percent = numeric(),
  Mean_Width_hg38 = numeric(),
  Mean_Width_hg19 = numeric(),
  Median_Width_hg38 = numeric(),
  Median_Width_hg19 = numeric(),
  stringsAsFactors = FALSE
)

# ------------------------------------------------
# 3. Run QC per cancer type
# ------------------------------------------------
for (ctype in cancer_types) {
  cat("=== Processing:", ctype, "===\n")

  hg38_file <- file.path(input_hg38_dir, paste0(ctype, "_peakCalls.txt"))
  hg19_file <- file.path(input_hg19_dir, paste0(ctype, "_ATAC_Peaks_hg19.bed"))

  if (!file.exists(hg38_file) || !file.exists(hg19_file)) {
    cat("  Skipping:", ctype, "(missing one of the files)\n\n")
    next
  }

  hg38_df <- fread(hg38_file)
  hg19_df <- fread(hg19_file)

  hg38_gr <- GRanges(seqnames = hg38_df$seqnames,
                     ranges = IRanges(start = hg38_df$start, end = hg38_df$end))
  hg19_gr <- GRanges(seqnames = hg19_df$Chrom,
                     ranges = IRanges(start = hg19_df$Start, end = hg19_df$End))

  # --- basic counts ---
  n38 <- length(hg38_gr)
  n19 <- length(hg19_gr)
  retention <- round(100 * n19 / n38, 2)

  cat("  hg38 regions:", n38, "\n")
  cat("  hg19 regions:", n19, "\n")
  cat("  Retention rate:", retention, "%\n")

  # --- width distributions ---
  w38_mean <- round(mean(width(hg38_gr)), 1)
  w38_med  <- median(width(hg38_gr))
  w19_mean <- round(mean(width(hg19_gr)), 1)
  w19_med  <- median(width(hg19_gr))
  cat("  Widths (mean/median):", w38_mean, "/", w38_med, "→", w19_mean, "/", w19_med, "\n")

  # --- chromosome distribution ---
  cat("  Chromosomes (hg19):\n")
  print(table(seqnames(hg19_gr)))

  # --- metadata check ---
  expected_cols <- c("name", "score", "annotation", "percentGC", "percentAT")
  meta_ok <- all(expected_cols %in% colnames(hg19_df))
  cat("  Metadata columns preserved:", meta_ok, "\n")

  # --- sample region sanity check ---
  cat("  Example region hg38:",
      as.character(seqnames(hg38_gr[1])), ":", start(hg38_gr[1]), "-", end(hg38_gr[1]), "\n")
  cat("  Example region hg19:",
      as.character(seqnames(hg19_gr[1])), ":", start(hg19_gr[1]), "-", end(hg19_gr[1]), "\n\n")

  # --- record summary ---
  qc_summary <- rbind(qc_summary, data.frame(
    Cancer_Type = ctype,
    hg38_Regions = n38,
    hg19_Regions = n19,
    Retention_Percent = retention,
    Mean_Width_hg38 = w38_mean,
    Mean_Width_hg19 = w19_mean,
    Median_Width_hg38 = w38_med,
    Median_Width_hg19 = w19_med
  ))
}




Found 23 cancer types for QC

=== Processing: ACC ===
  hg38 regions: 90777 
  hg19 regions: 90373 
  Retention rate: 99.55 %
  Widths (mean/median): 502 / 502 → 502 / 502 
  Chromosomes (hg19):

 chr1  chr2  chr3  chr4  chr5  chr6  chr7  chr8  chr9 chr10 chr11 chr12 chr13 
 7011  5911  5330  3695  6636  4732  5546  4502  3656  4084  3897  6368  1862 
chr14 chr15 chr16 chr17 chr18 chr19 chr20 chr21 chr22  chrX  chrY 
 3121  2159  4054  3616  1330  4971  3190   929  1372  2331    70 
  Metadata columns preserved: TRUE 
  Example region hg38: chr1 : 1291749 - 1292250 
  Example region hg19: chr1 : 1227129 - 1227630 

=== Processing: BLCA ===
  hg38 regions: 108473 
  hg19 regions: 107944 
  Retention rate: 99.51 %
  Widths (mean/median): 502 / 502 → 501.9 / 502 
  Chromosomes (hg19):

 chr1  chr2  chr3  chr4  chr5  chr6  chr7  chr8  chr9 chr10 chr11 chr12 chr13 
10948  7478  7396  4190  4717  5522  5964  4543  3773  4439  5074  5447  2617 
chr14 chr15 chr16 chr17 chr18 chr19 chr20 chr21 

## Older

### DHS Data

In [8]:
library(rtracklayer)
library(GenomicRanges)
library(data.table)

# Define paths
input_dir <- "/dcs07/scharpf/data/jzavras/LUCAS_Olink/Lucas-Cancer-Screening/scripts/lung-cancer-screening-paper/methods_code/WG/ref/PEARL/data/Bed_files_hg38/DHS_sites/"
output_dir <- "../ref/CRE_sites/DHS_sites"

chain <- import.chain("../ref/hg38ToHg19.over.chain")

# Get all BED files
bed_files <- list.files(input_dir, pattern = "\\.bed$", full.names = TRUE)

cat("Found", length(bed_files), "BED files to process\n\n")

# Initialize summary data frame
summary_stats <- data.frame(
    File = character(),
    Total_Regions = integer(),
    Successfully_Mapped = integer(),
    Failed_to_Map = integer(),
    Mapping_Rate_Percent = numeric(),
    stringsAsFactors = FALSE
)

# Process each file
for (bed_file in bed_files) {
    file_name <- basename(bed_file)
    cat("Processing:", file_name, "\n")
    
    # Read BED file
    dhs_df <- fread(bed_file, sep = "\t", header = TRUE, fill = TRUE)
    
    # Create GRanges
    dhs_gr <- GRanges(
        seqnames = dhs_df$Chrom,
        ranges   = IRanges(start = dhs_df$Start, end = dhs_df$End),
        strand   = "*"
    )
    mcols(dhs_gr) <- as.data.frame(dhs_df[, .(
        name, mean_signal, peak.count, summit, core_start, core_end,
        component, position, score
    )])
    
    # LiftOver
    lifted_list <- liftOver(dhs_gr, chain)
    success <- lengths(lifted_list) > 0
    
    # Calculate statistics
    total_regions <- length(dhs_gr)
    mapped <- sum(success)
    failed <- sum(!success)
    mapping_rate <- round(100 * mapped / total_regions, 2)
    
    # Print statistics
    cat("  Successfully mapped:", mapped, "\n")
    cat("  Failed to map:", failed, "\n")
    cat("  Mapping rate:", mapping_rate, "%\n")
    
    # Add to summary
    summary_stats <- rbind(summary_stats, data.frame(
        File = file_name,
        Total_Regions = total_regions,
        Successfully_Mapped = mapped,
        Failed_to_Map = failed,
        Mapping_Rate_Percent = mapping_rate
    ))
    
    # Get lifted regions
    lifted <- unlist(lifted_list)
    
    # Convert to data frame preserving all metadata
    lifted_df <- as.data.frame(lifted)
    
    # Reorder columns to match original BED format
    lifted_df <- lifted_df[, c("seqnames", "start", "end", 
                                "name", "mean_signal", "peak.count", "summit",
                                "core_start", "core_end", "component", 
                                "position", "score")]
    
    # Rename chromosome columns to match original
    colnames(lifted_df)[1:3] <- c("Chrom", "Start", "End")
    
    # Save with all metadata
    output_file <- file.path(output_dir, file_name)
    fwrite(lifted_df, output_file, sep = "\t", quote = FALSE)
    cat("  Saved to:", output_file, "\n\n")
}

# Save summary statistics
summary_file <- file.path(output_dir, "liftover_summary.txt")
fwrite(summary_stats, summary_file, sep = "\t", quote = FALSE)

cat("All files processed!\n")
cat("Summary statistics saved to:", summary_file, "\n")

# Print overall summary
cat("\n=== Overall Summary ===\n")
cat("Total files processed:", nrow(summary_stats), "\n")
cat("Total regions across all files:", sum(summary_stats$Total_Regions), "\n")
cat("Total successfully mapped:", sum(summary_stats$Successfully_Mapped), "\n")
cat("Total failed to map:", sum(summary_stats$Failed_to_Map), "\n")
cat("Overall mapping rate:", 
    round(100 * sum(summary_stats$Successfully_Mapped) / sum(summary_stats$Total_Regions), 2), 
    "%\n")


Found 16 BED files to process

Processing: Cancer_epithelial.bed 
  Successfully mapped: 126801 
  Failed to map: 408 
  Mapping rate: 99.68 %
  Saved to: ../ref/CRE_sites/DHS_sites/Cancer_epithelial.bed 

Processing: Cardiac.bed 
  Successfully mapped: 101442 
  Failed to map: 128 
  Mapping rate: 99.87 %
  Saved to: ../ref/CRE_sites/DHS_sites/Cardiac.bed 

Processing: Digestive.bed 
  Successfully mapped: 114287 
  Failed to map: 289 
  Mapping rate: 99.75 %
  Saved to: ../ref/CRE_sites/DHS_sites/Digestive.bed 

Processing: Lymphoid.bed 
  Successfully mapped: 191425 
  Failed to map: 947 
  Mapping rate: 99.51 %
  Saved to: ../ref/CRE_sites/DHS_sites/Lymphoid.bed 

Processing: Musculoskeletal.bed 
  Successfully mapped: 181508 
  Failed to map: 614 
  Mapping rate: 99.66 %
  Saved to: ../ref/CRE_sites/DHS_sites/Musculoskeletal.bed 

Processing: Myeloid_erythroid.bed 
  Successfully mapped: 144246 
  Failed to map: 708 
  Mapping rate: 99.51 %
  Saved to: ../ref/CRE_sites/DHS_sites/M

### scATAC Data

In [28]:
library(rtracklayer)
library(GenomicRanges)
library(data.table)

# Define paths
input_dir <- "/dcs07/scharpf/data/jzavras/LUCAS_Olink/Lucas-Cancer-Screening/scripts/lung-cancer-screening-paper/methods_code/WG/ref/PEARL/data/Bed_files_hg38/scATAC_sites/"
output_dir <- "../ref/CRE_sites/scATAC_sites/"

# Import chain file
chain <- import.chain("../ref/hg38ToHg19.over.chain")

# Get all BED files
bed_files <- list.files(input_dir, pattern = "\\.bed$", full.names = TRUE)

cat("Found", length(bed_files), "BED files to process\n\n")

# Initialize summary data frame
summary_stats <- data.frame(
    File = character(),
    Total_Regions = integer(),
    Successfully_Mapped = integer(),
    Failed_to_Map = integer(),
    Mapping_Rate_Percent = numeric(),
    stringsAsFactors = FALSE
)

# Process each file
for (bed_file in bed_files) {
    file_name <- basename(bed_file)
    cat("Processing:", file_name, "\n")
    
    # Read BED file
    atac_df <- fread(bed_file, sep = "\t", header = TRUE, fill = TRUE)
    
    # Rename reserved column names if they exist
    reserved_names <- c("seqnames", "ranges", "strand", "seqlevels", 
                        "seqlengths", "isCircular", "start", "end", 
                        "width", "element")
    
    for (reserved in reserved_names) {
        if (reserved %in% names(atac_df) && !(reserved %in% c("Chrom", "Start", "End"))) {
            new_name <- paste0(reserved, "_original")
            setnames(atac_df, reserved, new_name)
            cat("  Renamed column '", reserved, "' to '", new_name, "'\n", sep = "")
        }
    }
    
    # Create GRanges
    atac_gr <- GRanges(
        seqnames = atac_df$Chrom,
        ranges   = IRanges(start = atac_df$Start, end = atac_df$End),
        strand   = "*"
    )
    
    # Attach ALL metadata columns (excluding Chrom, Start, End)
    metadata_cols <- setdiff(names(atac_df), c("Chrom", "Start", "End"))
    
    mcols(atac_gr) <- as.data.frame(atac_df[, ..metadata_cols])
    
    # LiftOver
    lifted_list <- liftOver(atac_gr, chain)
    success <- lengths(lifted_list) > 0
    
    # Calculate statistics
    total_regions <- length(atac_gr)
    mapped <- sum(success)
    failed <- sum(!success)
    mapping_rate <- round(100 * mapped / total_regions, 2)
    
    # Print statistics
    cat("  Successfully mapped:", mapped, "\n")
    cat("  Failed to map:", failed, "\n")
    cat("  Mapping rate:", mapping_rate, "%\n")
    
    # Add to summary
    summary_stats <- rbind(summary_stats, data.frame(
        File = file_name,
        Total_Regions = total_regions,
        Successfully_Mapped = mapped,
        Failed_to_Map = failed,
        Mapping_Rate_Percent = mapping_rate
    ))
    
    # Get lifted regions
    lifted <- unlist(lifted_list)
    
    # Convert to data frame preserving all metadata
    lifted_df <- as.data.frame(lifted)
    
    # Reorder columns: coordinates first, then all metadata
    coord_cols <- c("seqnames", "start", "end")
    other_cols <- setdiff(names(lifted_df), c(coord_cols, "width", "strand"))
    lifted_df <- lifted_df[, c(coord_cols, other_cols)]
    
    # Rename chromosome columns to match original
    colnames(lifted_df)[1:3] <- c("Chrom", "Start", "End")
    
    # Rename back the reserved column names in output
    for (reserved in reserved_names) {
        new_name <- paste0(reserved, "_original")
        if (new_name %in% names(lifted_df)) {
            setnames(lifted_df, new_name, reserved)
        }
    }
    
    # Save with all metadata
    output_file <- file.path(output_dir, file_name)
    fwrite(lifted_df, output_file, sep = "\t", quote = FALSE)
    cat("  Saved to:", output_file, "\n\n")
}

# Save summary statistics
summary_file <- file.path(output_dir, "liftover_summary.txt")
fwrite(summary_stats, summary_file, sep = "\t", quote = FALSE)

cat("All files processed!\n")
cat("Summary statistics saved to:", summary_file, "\n")

# Print overall summary
cat("\n=== Overall Summary ===\n")
cat("Total files processed:", nrow(summary_stats), "\n")
cat("Total regions across all files:", sum(summary_stats$Total_Regions), "\n")
cat("Total successfully mapped:", sum(summary_stats$Successfully_Mapped), "\n")
cat("Total failed to map:", sum(summary_stats$Failed_to_Map), "\n")
cat("Overall mapping rate:", 
    round(100 * sum(summary_stats$Successfully_Mapped) / sum(summary_stats$Total_Regions), 2), 
    "%\n")

Found 20 BED files to process

Processing: Adrenal_Cortical.bed 
  Renamed column 'strand' to 'strand_original'


Discarding unchained sequences: chr7_KI270803v1_alt



  Successfully mapped: 36356 
  Failed to map: 33 
  Mapping rate: 99.91 %
  Saved to: ../ref/CRE_sites/scATAC_sites//Adrenal_Cortical.bed 

Processing: Adult_Stromal.bed 
  Renamed column 'strand' to 'strand_original'
  Successfully mapped: 70974 
  Failed to map: 889 
  Mapping rate: 98.76 %
  Saved to: ../ref/CRE_sites/scATAC_sites//Adult_Stromal.bed 

Processing: Cardiomyocyte.bed 
  Renamed column 'strand' to 'strand_original'


Discarding unchained sequences: chr22_KI270879v1_alt



  Successfully mapped: 22551 
  Failed to map: 13 
  Mapping rate: 99.94 %
  Saved to: ../ref/CRE_sites/scATAC_sites//Cardiomyocyte.bed 

Processing: Endothelial.bed 
  Renamed column 'strand' to 'strand_original'


...

  Successfully mapped: 36448 
  Failed to map: 34 
  Mapping rate: 99.91 %
  Saved to: ../ref/CRE_sites/scATAC_sites//Pulm_Epithelial.bed 

Processing: Skeletal_Myocyte.bed 
  Renamed column 'strand' to 'strand_original'


Discarding unchained sequences: chr7_KI270803v1_alt



  Successfully mapped: 27904 
  Failed to map: 15 
  Mapping rate: 99.95 %
  Saved to: ../ref/CRE_sites/scATAC_sites//Skeletal_Myocyte.bed 

Processing: Skin_Mammary_Epithelial.bed 
  Renamed column 'strand' to 'strand_original'
  Successfully mapped: 48267 
  Failed to map: 69 
  Mapping rate: 99.86 %
  Saved to: ../ref/CRE_sites/scATAC_sites//Skin_Mammary_Epithelial.bed 

Processing: Stromal.bed 
  Renamed column 'strand' to 'strand_original'


Discarding unchained sequences: chr22_KI270879v1_alt, chr7_KI270803v1_alt, chr8_KI270821v1_alt, chr15_KI270850v1_alt, chr4_GL000008v2_random, chrUn_KI270742v1



  Successfully mapped: 62005 
  Failed to map: 98 
  Mapping rate: 99.84 %
  Saved to: ../ref/CRE_sites/scATAC_sites//Stromal.bed 

Processing: Ubiquitous.bed 
  Renamed column 'strand' to 'strand_original'
  Successfully mapped: 126271 
  Failed to map: 0 
  Mapping rate: 100 %
  Saved to: ../ref/CRE_sites/scATAC_sites//Ubiquitous.bed 

All files processed!
Summary statistics saved to: ../ref/CRE_sites/scATAC_sites//liftover_summary.txt 

=== Overall Summary ===
Total files processed: 20 
Total regions across all files: 1154611 
Total successfully mapped: 1152846 
Total failed to map: 1765 
Overall mapping rate: 99.85 %


### TCGA Data

In [1]:
#!/usr/bin/env Rscript
# ===================================================================
# Liftover TCGA ATAC-seq Cancer Type Peak Files (hg38 → hg19)
# Outputs: .bed files with metadata for cfDNA WGS-CRE footprinting
# ===================================================================

suppressPackageStartupMessages({
  library(rtracklayer)
  library(GenomicRanges)
  library(data.table)
})

# -----------------------------------------------------------------
# Paths
# -----------------------------------------------------------------
input_dir  <- "../ref/TCGA/"
output_dir <- "../ref/CRE_sites/TCGA_sites2/"
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

# Import hg38 → hg19 chain
chain <- import.chain("../ref/hg38ToHg19.over.chain")

# Get all TCGA cancer-specific .txt files
txt_files <- list.files(input_dir, pattern = "\\.txt$", full.names = TRUE)
cat("Found", length(txt_files), "files to process\n\n")

# -----------------------------------------------------------------
# Initialize summary statistics
# -----------------------------------------------------------------
summary_stats <- data.frame(
  File = character(),
  Total_Regions = integer(),
  Successfully_Mapped = integer(),
  Failed_to_Map = integer(),
  Mapping_Rate_Percent = numeric(),
  stringsAsFactors = FALSE
)

# -----------------------------------------------------------------
# Process each .txt file
# -----------------------------------------------------------------
for (txt_file in txt_files) {
  file_name <- basename(txt_file)
  cancer_type <- sub("_ATAC.*", "", file_name)
  cat("Processing:", file_name, "\n")

  # Load file
  df <- fread(txt_file, sep = "\t", header = TRUE)
  stopifnot(all(c("seqnames", "start", "end") %in% names(df)))

  # Create GRanges
  gr <- GRanges(
    seqnames = df$seqnames,
    ranges   = IRanges(start = df$start, end = df$end),
    strand   = "*"
  )

  # Add metadata
  meta_cols <- setdiff(names(df), c("seqnames", "start", "end"))
  mcols(gr) <- as.data.frame(df[, ..meta_cols])

  # Liftover
  lifted_list <- liftOver(gr, chain)
  success <- lengths(lifted_list) > 0
  total_regions <- length(gr)
  mapped <- sum(success)
  failed <- total_regions - mapped
  rate <- round(100 * mapped / total_regions, 2)

  cat("  Successfully mapped:", mapped, "\n")
  cat("  Failed:", failed, "\n")
  cat("  Mapping rate:", rate, "%\n")

  summary_stats <- rbind(summary_stats, data.frame(
    File = file_name,
    Total_Regions = total_regions,
    Successfully_Mapped = mapped,
    Failed_to_Map = failed,
    Mapping_Rate_Percent = rate
  ))

  # Flatten lifted list
  lifted <- unlist(lifted_list)
  genome(lifted) <- "hg19"

  # Build BED dataframe with metadata
  lifted_df <- data.frame(
    chrom = as.character(seqnames(lifted)),
    start = start(lifted),
    end = end(lifted),
    name = if ("name" %in% colnames(mcols(lifted))) mcols(lifted)$name else paste0(cancer_type, "_", seq_along(lifted)),
    score = if ("score" %in% colnames(mcols(lifted))) mcols(lifted)$score else ".",
    annotation = if ("annotation" %in% colnames(mcols(lifted))) mcols(lifted)$annotation else ".",
    percentGC = if ("percentGC" %in% colnames(mcols(lifted))) mcols(lifted)$percentGC else NA,
    percentAT = if ("percentAT" %in% colnames(mcols(lifted))) mcols(lifted)$percentAT else NA
  )

  # Save to new directory
  out_bed <- file.path(output_dir, paste0(cancer_type, "_ATAC_Peaks_hg19.bed"))
  fwrite(lifted_df, out_bed, sep = "\t", quote = FALSE, col.names = TRUE)
  cat("  Saved:", out_bed, "\n\n")
}

# -----------------------------------------------------------------
# Save liftover summary
# -----------------------------------------------------------------
summary_file <- file.path(output_dir, "liftover_summary.txt")
fwrite(summary_stats, summary_file, sep = "\t", quote = FALSE)

cat("\n=== Liftover Complete ===\n")
cat("Output directory:", output_dir, "\n")
cat("Summary saved to:", summary_file, "\n")
cat("Total files processed:", nrow(summary_stats), "\n")
cat("Overall mapping rate:",
    round(100 * sum(summary_stats$Successfully_Mapped) /
          sum(summary_stats$Total_Regions), 2), "%\n")


Warning message:
“package ‘rtracklayer’ was built under R version 4.3.3”
Warning message:
“package ‘GenomicRanges’ was built under R version 4.3.3”
Warning message:
“package ‘BiocGenerics’ was built under R version 4.3.2”
Warning message:
“package ‘S4Vectors’ was built under R version 4.3.3”
Warning message:
“package ‘IRanges’ was built under R version 4.3.3”
Warning message:
“package ‘GenomeInfoDb’ was built under R version 4.3.2”
Warning message:
“package ‘data.table’ was built under R version 4.3.3”


Found 23 files to process

Processing: ACC_peakCalls.txt 
  Successfully mapped: 90663 
  Failed: 114 
  Mapping rate: 99.87 %
  Saved: ../ref/CRE_sites/TCGA_sites2//ACC_peakCalls.txt_ATAC_Peaks_hg19.bed 

Processing: BLCA_peakCalls.txt 
  Successfully mapped: 108298 
  Failed: 175 
  Mapping rate: 99.84 %
  Saved: ../ref/CRE_sites/TCGA_sites2//BLCA_peakCalls.txt_ATAC_Peaks_hg19.bed 

Processing: BRCA_peakCalls.txt 
  Successfully mapped: 215468 
  Failed: 510 
  Mapping rate: 99.76 %
  Saved: ../ref/CRE_sites/TCGA_sites2//BRCA_peakCalls.txt_ATAC_Peaks_hg19.bed 

Processing: CESC_peakCalls.txt 
  Successfully mapped: 56022 
  Failed: 103 
  Mapping rate: 99.82 %
  Saved: ../ref/CRE_sites/TCGA_sites2//CESC_peakCalls.txt_ATAC_Peaks_hg19.bed 

Processing: CHOL_peakCalls.txt 
  Successfully mapped: 67977 
  Failed: 149 
  Mapping rate: 99.78 %
  Saved: ../ref/CRE_sites/TCGA_sites2//CHOL_peakCalls.txt_ATAC_Peaks_hg19.bed 

Processing: COAD_peakCalls.txt 
  Successfully mapped: 122765 
  Fai

### QC

In [30]:
##### QC of the Liftover #####

library(rtracklayer)
library(GenomicRanges)
library(data.table)

# Load the lifted ATAC-seq file (hg19)
atac_hg19_file <- "../ref/CRE_sites/scATAC_sites/Adrenal_Cortical.bed"  # change filename as needed
atac_hg19_df <- fread(atac_hg19_file, sep = "\t", header = TRUE)
atac_hg19_gr <- GRanges(
    seqnames = atac_hg19_df$Chrom,
    ranges = IRanges(start = atac_hg19_df$Start, end = atac_hg19_df$End)
)

# Load the lifted DHS file (hg19)
dhs_hg19_file <- "../ref/CRE_sites/DHS_sites/Cardiac.bed"  # change filename as needed
dhs_hg19_df <- fread(dhs_hg19_file, sep = "\t", header = TRUE)
dhs_hg19_gr <- GRanges(
    seqnames = dhs_hg19_df$Chrom,
    ranges = IRanges(start = dhs_hg19_df$Start, end = dhs_hg19_df$End)
)

# Load the original hg38 files for comparison
atac_hg38_file <- "/dcs07/scharpf/data/jzavras/LUCAS_Olink/Lucas-Cancer-Screening/scripts/lung-cancer-screening-paper/methods_code/WG/ref/PEARL/data/Bed_files_hg38/scATAC_sites/Adrenal_Cortical.bed"  # change path as needed
atac_hg38_df <- fread(atac_hg38_file, sep = "\t", header = TRUE)
atac_hg38_gr <- GRanges(
    seqnames = atac_hg38_df$Chrom,
    ranges = IRanges(start = atac_hg38_df$Start, end = atac_hg38_df$End)
)

dhs_hg38_file <- "/dcs07/scharpf/data/jzavras/LUCAS_Olink/Lucas-Cancer-Screening/scripts/lung-cancer-screening-paper/methods_code/WG/ref/PEARL/data/Bed_files_hg38/DHS_sites/Cardiac.bed"  # change path as needed
dhs_hg38_df <- fread(dhs_hg38_file, sep = "\t", header = TRUE)
dhs_hg38_gr <- GRanges(
    seqnames = dhs_hg38_df$Chrom,
    ranges = IRanges(start = dhs_hg38_df$Start, end = dhs_hg38_df$End)
)

cat("=== BASIC STATISTICS ===\n\n")

# Check number of regions before and after
cat("ATAC-seq:\n")
cat("  hg38 regions:", length(atac_hg38_gr), "\n")
cat("  hg19 regions:", length(atac_hg19_gr), "\n")
cat("  Regions lost:", length(atac_hg38_gr) - length(atac_hg19_gr), "\n")
cat("  Retention rate:", round(100 * length(atac_hg19_gr) / length(atac_hg38_gr), 2), "%\n\n")

cat("DHS:\n")
cat("  hg38 regions:", length(dhs_hg38_gr), "\n")
cat("  hg19 regions:", length(dhs_hg19_gr), "\n")
cat("  Regions lost:", length(dhs_hg38_gr) - length(dhs_hg19_gr), "\n")
cat("  Retention rate:", round(100 * length(dhs_hg19_gr) / length(dhs_hg38_gr), 2), "%\n\n")

cat("=== CHROMOSOME DISTRIBUTION ===\n\n")

# Check chromosome distribution
cat("ATAC-seq hg19 - Regions per chromosome:\n")
print(table(seqnames(atac_hg19_gr)))
cat("\n")

cat("DHS hg19 - Regions per chromosome:\n")
print(table(seqnames(dhs_hg19_gr)))
cat("\n")

cat("=== COORDINATE SHIFT ANALYSIS ===\n\n")

# For regions that mapped, calculate the coordinate shift
# Sample a few regions to check
set.seed(123)
sample_indices <- sample(1:min(100, length(atac_hg38_gr), length(atac_hg19_gr)), 
                         min(10, length(atac_hg19_gr)))

cat("Sample coordinate shifts (ATAC-seq, first 10 regions):\n")
cat("hg38 -> hg19\n")
for (i in sample_indices) {
    hg38_start <- start(atac_hg38_gr[i])
    hg19_start <- start(atac_hg19_gr[i])
    shift <- hg19_start - hg38_start
    cat(sprintf("  Region %d: chr%s:%d -> chr%s:%d (shift: %+d bp)\n",
                i, 
                as.character(seqnames(atac_hg38_gr[i])), hg38_start,
                as.character(seqnames(atac_hg19_gr[i])), hg19_start,
                shift))
}

cat("\n=== WIDTH DISTRIBUTION ===\n\n")

# Check if region widths are preserved
cat("ATAC-seq region widths:\n")
cat("  hg38 - Mean:", round(mean(width(atac_hg38_gr)), 1), 
    "bp, Median:", median(width(atac_hg38_gr)), "bp\n")
cat("  hg19 - Mean:", round(mean(width(atac_hg19_gr)), 1), 
    "bp, Median:", median(width(atac_hg19_gr)), "bp\n\n")

cat("DHS region widths:\n")
cat("  hg38 - Mean:", round(mean(width(dhs_hg38_gr)), 1), 
    "bp, Median:", median(width(dhs_hg38_gr)), "bp\n")
cat("  hg19 - Mean:", round(mean(width(dhs_hg19_gr)), 1), 
    "bp, Median:", median(width(dhs_hg19_gr)), "bp\n\n")

cat("=== METADATA PRESERVATION CHECK ===\n\n")

# Check that metadata was preserved
cat("ATAC-seq metadata columns:\n")
cat("  hg38:", paste(names(atac_hg38_df), collapse=", "), "\n")
cat("  hg19:", paste(names(atac_hg19_df), collapse=", "), "\n")
cat("  Columns preserved:", all(names(atac_hg38_df) %in% names(atac_hg19_df)), "\n\n")

cat("DHS metadata columns:\n")
cat("  hg38:", paste(names(dhs_hg38_df), collapse=", "), "\n")
cat("  hg19:", paste(names(dhs_hg19_df), collapse=", "), "\n")
cat("  Columns preserved:", all(names(dhs_hg38_df) %in% names(dhs_hg19_df)), "\n\n")

# Check a specific region in detail
cat("=== DETAILED CHECK OF FIRST REGION ===\n\n")
cat("ATAC-seq first region:\n")
cat("  hg38:", as.character(seqnames(atac_hg38_gr[1])), ":", 
    start(atac_hg38_gr[1]), "-", end(atac_hg38_gr[1]), "\n")
cat("  hg19:", as.character(seqnames(atac_hg19_gr[1])), ":", 
    start(atac_hg19_gr[1]), "-", end(atac_hg19_gr[1]), "\n")
cat("  Width change:", width(atac_hg38_gr[1]), "->", width(atac_hg19_gr[1]), "\n")
if ("score" %in% names(atac_hg38_df) && "score" %in% names(atac_hg19_df)) {
    cat("  Score preserved:", atac_hg38_df$score[1], "->", atac_hg19_df$score[1], "\n")
}

cat("\nDHS first region:\n")
cat("  hg38:", as.character(seqnames(dhs_hg38_gr[1])), ":", 
    start(dhs_hg38_gr[1]), "-", end(dhs_hg38_gr[1]), "\n")
cat("  hg19:", as.character(seqnames(dhs_hg19_gr[1])), ":", 
    start(dhs_hg19_gr[1]), "-", end(dhs_hg19_gr[1]), "\n")
cat("  Width change:", width(dhs_hg38_gr[1]), "->", width(dhs_hg19_gr[1]), "\n")
if ("score" %in% names(dhs_hg38_df) && "score" %in% names(dhs_hg19_df)) {
    cat("  Score preserved:", dhs_hg38_df$score[1], "->", dhs_hg19_df$score[1], "\n")
}

cat("\n=== QUALITY CONTROL SUMMARY ===\n\n")

# Overall QC summary
qc_pass <- TRUE
issues <- c()

if (length(atac_hg19_gr) < 0.95 * length(atac_hg38_gr)) {
    issues <- c(issues, "ATAC: >5% of regions lost")
    qc_pass <- FALSE
}

if (length(dhs_hg19_gr) < 0.95 * length(dhs_hg38_gr)) {
    issues <- c(issues, "DHS: >5% of regions lost")
    qc_pass <- FALSE
}

if (!all(names(atac_hg38_df) %in% names(atac_hg19_df))) {
    issues <- c(issues, "ATAC: Some metadata columns lost")
    qc_pass <- FALSE
}

if (!all(names(dhs_hg38_df) %in% names(dhs_hg19_df))) {
    issues <- c(issues, "DHS: Some metadata columns lost")
    qc_pass <- FALSE
}

if (qc_pass) {
    cat("✓ QC PASSED: LiftOver completed successfully!\n")
    cat("  - High retention rates (>95%)\n")
    cat("  - All metadata preserved\n")
    cat("  - Region widths maintained\n")
} else {
    cat("✗ QC ISSUES DETECTED:\n")
    for (issue in issues) {
        cat("  -", issue, "\n")
    }
}

=== BASIC STATISTICS ===

ATAC-seq:
  hg38 regions: 36389 
  hg19 regions: 36479 
  Regions lost: -90 
  Retention rate: 100.25 %

DHS:
  hg38 regions: 101570 
  hg19 regions: 101592 
  Regions lost: -22 
  Retention rate: 100.02 %

=== CHROMOSOME DISTRIBUTION ===

ATAC-seq hg19 - Regions per chromosome:

 chr4  chrX  chr2  chr1 chr11 chr10  chr3 chr16 chr12  chr7  chr5 chr19 chr13 
 1659  1070  3090  3269  1797  1651  2526  1274  1850  1937  1883   936  1109 
chr20  chr9  chr6 chr22 chr17 chr15  chr8 chr18 chr14 chr21 
 1114  1670  1975   724  1400  1170  1794   898  1244   439 

DHS hg19 - Regions per chromosome:

 chr1 chr10 chr11 chr12 chr13 chr14 chr15 chr16 chr17 chr18 chr19  chr2 chr20 
 9369  5325  4848  4731  3383  2889  3515  2422  2846  2918  1478  8843  2075 
chr21 chr22  chr3  chr4  chr5  chr6  chr7  chr8  chr9  chrX  chrY 
 1036  1392  7638  6439  6661  6545  5418  5156  3910  2651   104 

=== COORDINATE SHIFT ANALYSIS ===

Sample coordinate shifts (ATAC-seq, first 10 reg